In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# Constrained docking

Dock a ligand while holding MCS-matched atoms near a reference pose.

We treat one BRD ligand SDF as an already-docked **reference pose** (3D coordinates
from the file) and constrained-dock a second, similar BRD ligand aligned to it via
`ConstrainedDocking(..., reference=...)`.

## Setup

In [ ]:
import math
from pathlib import Path

from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    ConstrainedDocking,
    Ligand,
    Pocket,
    Protein,
)

from deeporigin.platform import DeepOriginClient


In [ ]:
client = DeepOriginClient()
client

## Load protein

Sync the BRD protein to the platform so the docking tools can reference it by ID.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.remove_water()
protein.sync()
protein.id

## Reference pose

Load a ligand from disk and treat it as a docked reference pose. In a real workflow
this would come from `Docking.run()`; here we use `brd-2.sdf` directly because it
already has 3D coordinates.

In [ ]:
reference_pose = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
reference_pose

## Query ligand

Load a similar ligand to dock. Constrained docking requires a structure file on the
platform so constraint atom indices match the uploaded molecule.

In [ ]:
query_ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-3.sdf")
query_ligand.sync()
query_ligand

## Binding pocket

Load the BRD pocket from the repo test fixture (`tests/fixtures/files/pocketfinder/pocket_1.pdb`) instead of running PocketFinder.

In [ ]:
pf = PocketFinder(protein=protein, pocket_count=1)
pockets = pf.run()
protein.show(pockets=pockets)

## Constrained docking

Pass the reference pose to derive harmonic constraints from the maximum common
substructure (MCS). Only the query ligand is sent to the tool to be docked.

In [ ]:
cd = ConstrainedDocking(
    protein=protein,
    pocket=pockets[0],
    ligand=query_ligand,
    reference=reference_pose,
    effort=5,
)
cd

## Estimate cost

In [ ]:
cd.run(quote=True)
cd.estimate

## Run

Constrained docking is synchronous: `run()` blocks until poses are returned.

In [ ]:
poses = cd.run()
poses

In [ ]:
poses.to_dataframe()

## Visualize

Show the new poses together with the reference pose in the binding site.

In [ ]:
poses.download()
protein.show(poses=reference_pose + poses)